[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tuesdaythe13th/artifex-warp-newton-sim/blob/main/artifex_disc_cool_3d.ipynb)


# ARTIFEX DISC COOLING (Advanced 3D)
Simulating thin disc cooling and crystallinity with NVIDIA Warp.

In [ ]:
!pip install warp-lang numpy plotly

In [ ]:
import warp as wp
import warp.fem as fem
import numpy as np
import plotly.graph_objects as go

# ---------------------------------------------------------------
# Artifex Disc Parameters
# ---------------------------------------------------------------
DISC_DIAMETER = 0.305       # meters (12-inch LP)
DISC_THICKNESS = 0.0019     # meters (1.9 mm)
MELT_TEMP = 270.0           # °C
MOLD_TEMP = 95.0            # °C
T_G = 75.0                  # Glass transition °C
COOLING_TIME = 20.0         # seconds
THERMAL_CONDUCTIVITY = 0.20 # W/m·K (PET)
DENSITY = 1350              # kg/m³
SPECIFIC_HEAT = 1200        # J/kg·K
THERMAL_DIFFUSIVITY = THERMAL_CONDUCTIVITY / (DENSITY * SPECIFIC_HEAT)

# Crystallization parameters (simplified Avrami)
CRYSTAL_PEAK_TEMP = 160.0   # °C where crystallization is fastest
MAX_CRYSTAL_RATE = 0.001    # per second at peak (simplified)

wp.init()

In [ ]:
# ---------------------------------------------------------------
# Geometry: Thin 3D disc grid
# ---------------------------------------------------------------
reso_radial = 48
reso_angular = 48
reso_thick = 4

grid = fem.Grid3D(
    res=wp.vec3i(reso_radial, reso_angular, reso_thick),
    bounds_lo=wp.vec3(-DISC_DIAMETER/2, -DISC_DIAMETER/2, 0.0),
    bounds_hi=wp.vec3(DISC_DIAMETER/2, DISC_DIAMETER/2, DISC_THICKNESS)
)

scalar_space = fem.make_polynomial_space(grid, degree=1)

@fem.integrand
def diffusion_form(s: fem.Sample, u: fem.Field, v: fem.Field, k: float):
    return k * wp.dot(fem.grad(u, s), fem.grad(v, s))

@wp.kernel
def crystallinity_step(
    temp: wp.array(dtype=float),
    crystal: wp.array(dtype=float),
    dt: float,
    peak_temp: float,
    max_rate: float,
    Tg: float
):
    tid = wp.tid()
    T = temp[tid]
    chi = crystal[tid]
    if T < Tg:
        rate = 0.0
    else:
        sigma = 40.0
        temp_factor = wp.exp(-(T - peak_temp) * (T - peak_temp) / (2.0 * sigma * sigma))
        rate = max_rate * temp_factor * (1.0 - chi)
    crystal[tid] = chi + rate * dt

@wp.kernel
def update_temp_kernel(
    temp: wp.array(dtype=float),
    residual: wp.array(dtype=float),
    M_diag: float,
    dt: float,
    mold_temp: float,
    reso_radial: int,
    reso_angular: int,
    reso_thick: int
):
    tid = wp.tid()
    layer_size = (reso_radial + 1) * (reso_angular + 1)
    z_idx = tid // layer_size
    if z_idx == 0 or z_idx == reso_thick:
        temp[tid] = mold_temp
    else:
        temp[tid] -= dt * residual[tid] / M_diag


In [ ]:
def solve_cooling(n_steps: int = 150):
    dt = COOLING_TIME / n_steps
    temp_array = wp.full(scalar_space.node_count(), MELT_TEMP, dtype=float)
    crystal_array = wp.zeros(scalar_space.node_count(), dtype=float)
    
    domain = fem.Cells(geometry=grid)
    test = fem.make_test(space=scalar_space, domain=domain)
    
    node_vol = float((DISC_DIAMETER/reso_radial) * (DISC_DIAMETER/reso_angular) * (DISC_THICKNESS/reso_thick))
    M_diag_val = float(DENSITY * SPECIFIC_HEAT * node_vol)
    
    temp_field = scalar_space.make_field() 
    
    print(f"Executing {n_steps} steps...")
    for step in range(n_steps):
        temp_field.dof_values = temp_array
        try:
            residual = fem.integrate(
                diffusion_form,
                fields={"u": temp_field, "v": test},
                values={"k": THERMAL_CONDUCTIVITY},
                output_dtype=wp.float32
            )
        except Exception as e:
            print("Reverting to float:", e)
            residual = fem.integrate(
                diffusion_form,
                fields={"u": temp_field, "v": test},
                values={"k": THERMAL_CONDUCTIVITY},
                output_dtype=float
            )
            
        wp.launch(update_temp_kernel, dim=scalar_space.node_count(), 
                  inputs=[temp_array, residual, M_diag_val, dt, MOLD_TEMP, reso_radial, reso_angular, reso_thick])
        wp.launch(crystallinity_step, dim=scalar_space.node_count(), 
                  inputs=[temp_array, crystal_array, dt, CRYSTAL_PEAK_TEMP, MAX_CRYSTAL_RATE, T_G])
        
        if step > 0 and step % 30 == 0:
             print(f"  Step {step:3d}: max T={float(wp.max(temp_array)):.1f}°C, max χ={float(wp.max(crystal_array)):.4f}")
        
    return temp_array, crystal_array


In [ ]:
temp_final, crystal_final = solve_cooling(n_steps=150)

max_temp = float(wp.max(temp_final))
min_temp = float(wp.min(temp_final))
max_crystal = float(wp.max(crystal_final))
mean_crystal = float(wp.sum(crystal_final)) / crystal_final.shape[0]

print("\n" + "="*50)
print("ARTIFEX DISC COOLING SIMULATION RESULTS")
print("="*50)
print(f"Final max temperature: {max_temp:.1f}°C")
print(f"Final min temperature: {min_temp:.1f}°C")
print(f"Max crystallinity: {max_crystal:.4f} ({max_crystal*100:.2f}%)")
print(f"Mean crystallinity: {mean_crystal:.4f} ({mean_crystal*100:.2f}%)")
print(f"Target: < 5% crystallinity for acoustic transparency")

if max_crystal < 0.05:
    print("✓ PASS: Crystallinity target met")
else:
    print("✗ FAIL: Excessive crystallinity - reduce cooling time or mold temp")

### 3D Visualization
We use **Plotly** to render an interactive 3D scatter plot of the record disc's final crystallinity distribution.

In [ ]:
# Get coordinates directly from the FEM space
coords = scalar_space.node_positions().numpy()
x, y, z = coords[:, 0], coords[:, 1], coords[:, 2]

# Filter down to points that are strictly within the disc radius bounds to make it look like a cylinder
radius_sq = x**2 + y**2
max_radius_sq = (DISC_DIAMETER/2)**2
valid_mask = radius_sq <= max_radius_sq

x_v = x[valid_mask]
y_v = y[valid_mask]
z_v = z[valid_mask]
val_v = crystal_final.numpy()[valid_mask]

# Subsample if too large for browser rendering
sub_rate = max(1, len(x_v) // 15000)

fig = go.Figure(data=[go.Scatter3d(
    x=x_v[::sub_rate], 
    y=y_v[::sub_rate], 
    z=z_v[::sub_rate],
    mode='markers',
    marker=dict(
        size=3,
        color=val_v[::sub_rate],
        colorscale='Viridis',
        colorbar=dict(title="Crystallinity"),
        opacity=0.8
    )
)])

fig.update_layout(
    title="Artifex Post-Injection Crystallinity 3D (Warp + Plotly)",
    scene=dict(
        xaxis_title="X (m)", 
        yaxis_title="Y (m)", 
        zaxis_title="Z (m)",
        aspectmode='data' 
    ),
    margin=dict(l=0, r=0, b=0, t=50)
)

fig.show()
